# Tests: `fasterai.regularize.group_regularize_callback` (source `nbs/regularize/group_regularize_callback.ipynb`)

In [ ]:
from fastcore.test import *
from copy import deepcopy
import tempfile
import torch
import torch.nn as nn
from torch.amp import GradScaler
from torch.utils.data import TensorDataset
from fastai.callback.core import Callback
from fastai.callback.fp16 import MixedPrecision, NonNativeMixedPrecision
from fastai.callback.training import GradientAccumulation
from fastai.data.core import DataLoaders
from fastai.learner import Learner
from fastai.optimizer import SGD
from fasterai.core.criteria import large_final
from fasterai.core.schedule import lin, one_shot
from fasterai.prune.pruner import Pruner
from fasterai.prune.prune_callback import PruneCallback
from fasterai.regularize.group_regularize_callback import *

In [ ]:
# --- the penalty reaches the gradients through a real fit step ---
# values from test_pruner's `_Tiny`, reg 0.01; a zero data loss and lr 0 leave only the penalty
class _Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv, self.bn, self.fc = nn.Conv2d(1, 2, 1, bias=False), nn.BatchNorm2d(2), nn.Linear(2, 1)
        with torch.no_grad(): self.conv.weight.copy_(torch.tensor([1., 4.]).view(2, 1, 1, 1)); self.fc.weight.fill_(1.)
    def forward(self, x): return self.fc(self.bn(self.conv(x)).mean((2, 3)))

_EXPECTED = {'conv.weight': torch.tensor([0.08, 0.04]), 'bn.weight': torch.tensor([0.08, 0.01]), 'fc.weight': torch.tensor([0.08, 0.01])}

class _Probe(Callback):
    "Record the gradients the optimizer is about to see, at `order`"
    def __init__(self, order=60): self.order, self.grads = order, []
    def before_step(self): self.grads.append({n: p.grad.detach().flatten().clone() for n, p in self.learn.model.named_parameters()})

def _fit(learn, *cbs, n=1, **kw):
    with learn.no_bar(), learn.no_logging(): learn.fit(n, cbs=list(cbs), **kw)

def _tiny_learner(n=8, bs=2, reg=0.01):
    m = _Tiny()
    X, y = torch.randn(n, 1, 2, 2), torch.zeros(n, 1)
    dls = DataLoaders.from_dsets(TensorDataset(X, y), TensorDataset(X[:bs], y[:bs]), bs=bs, device='cpu')
    learn = Learner(dls, m, loss_func=lambda p, y: p.sum() * 0, opt_func=SGD, lr=0.)
    return learn, Pruner(m, 0.5, 'local', large_final, example_inputs=X[:bs], reg=reg, alpha=3)

def _run(*cbs, schedule=None, probe=None, **learner_kw):
    "Fit a fresh `_Tiny` learner with the callback, `cbs` and a probe; return the probe"
    learn, pr = _tiny_learner(**learner_kw)
    probe = probe or _Probe()
    _fit(learn, GroupRegularizeCallback(pr, schedule=schedule), *cbs, probe)
    return probe

def _check(g, factor=1.):
    for n, e in _EXPECTED.items(): test_close(g[n], factor * e, eps=1e-6)
    test_eq(g['bn.bias'].abs().max().item(), 0.); test_eq(g['fc.bias'].abs().max().item(), 0.)

probe = _run()
test_eq(len(probe.grads), 4)
for g in probe.grads: _check(g)

# without the callback, the same fit sees zero gradients
learn, pr = _tiny_learner(); probe = _Probe()
_fit(learn, probe)
assert all(t.abs().max() == 0 for t in probe.grads[0].values())

In [ ]:
# --- order: strictly after GradientAccumulation, strictly before MixedPrecision (which unscales) ---
assert GradientAccumulation.order < GroupRegularizeCallback.order < MixedPrecision.order

In [ ]:
# --- mixed precision: a scaled gradient gets a scaled penalty, the optimizer sees the true one ---
# CPU stub of MixedPrecision (order 10): a real GradScaler scales the loss, the gradients are divided back at the step
S = 1024.
class _Scaled(Callback):
    order = MixedPrecision.order
    def before_fit(self): self.learn.scaler = GradScaler('cpu', init_scale=S)
    def before_backward(self): self.learn.loss_grad = self.learn.scaler.scale(self.learn.loss_grad)
    def before_step(self):
        for p in self.learn.model.parameters(): p.grad /= S
    def after_fit(self): self.learn.scaler = None

before = _Probe(order=MixedPrecision.order - 1)
after = _run(_Scaled(), before)
for gb, ga in zip(before.grads, after.grads): _check(gb, S); _check(ga)

# only a real GradScaler is read: `learn.scaler` falls through to the model, whose attribute is ignored
learn, pr = _tiny_learner()
learn.model.scaler = 7.
probe = _Probe()
_fit(learn, GroupRegularizeCallback(pr), probe)
_check(probe.grads[0])

In [ ]:
# --- gradient accumulation: one penalty per real optimizer step, not per batch ---
learn, pr = _tiny_learner(n=8, bs=2)
calls, _reg = [], pr.regularize
pr.regularize = lambda scale=1.: (calls.append(scale), _reg(scale))
probe = _Probe()
_fit(learn, GroupRegularizeCallback(pr), GradientAccumulation(n_acc=4), probe)
test_eq(len(calls), 2); test_eq(len(probe.grads), 2)
for g in probe.grads: _check(g)

In [ ]:
# --- a schedule ramps the penalty, and the shared module-level schedule is left alone ---
for sched, progress in ((one_shot, [0, 0, 1, 1]), (lin, [0, .25, .5, .75])):
    state = deepcopy(sched.__dict__)
    probe = _run(schedule=sched)
    for g, f in zip(probe.grads, progress): _check(g, f)
    test_eq(sched.__dict__, state)
assert GroupRegularizeCallback(_tiny_learner()[1], schedule=lin).schedule is not lin

# verbose prints the penalty reached at the end of each epoch
learn, pr = _tiny_learner()
test_stdout(lambda: _fit(learn, GroupRegularizeCallback(pr, schedule=lin, verbose=True)), 'Group penalty: 7.50e-03')

In [ ]:
# --- refusals, each with one line naming the fix ---
learn, pr = _tiny_learner()
with ExceptionExpected(ValueError, regex='reg > 0'): GroupRegularizeCallback(_tiny_learner(reg=0.)[1])

_other = _tiny_learner()[1]   # a Pruner built on another model
with ExceptionExpected(ValueError, regex='another model than learn.model'): _fit(learn, GroupRegularizeCallback(_other))
with ExceptionExpected(ValueError, regex='use MixedPrecision'): _fit(learn, GroupRegularizeCallback(pr), NonNativeMixedPrecision())
with ExceptionExpected(ValueError, regex='PruneCallback cannot run in the same fit'):
    _fit(learn, GroupRegularizeCallback(pr), PruneCallback(0.5, one_shot, 'local', large_final, example_inputs=torch.randn(2, 1, 2, 2)))

# a refused fit leaves no callback behind: the next fit runs
_fit(learn, GroupRegularizeCallback(pr))
assert not any(isinstance(c, GroupRegularizeCallback) for c in learn.cbs)

In [ ]:
# --- the documented protocol: fit with the callback, prune after it, fine-tune with reset_opt=True ---
_live = lambda model: {id(p) for p in model.parameters()}
_held = lambda learn: {id(p) for g in learn.opt.param_lists for p in g}

torch.manual_seed(0)
_model = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
                       nn.Conv2d(8, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
                       nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 4))
_X, _y = torch.randn(32, 3, 8, 8), torch.randint(0, 4, (32,))
_dls = DataLoaders.from_dsets(TensorDataset(_X[:24], _y[:24]), TensorDataset(_X[24:], _y[24:]), bs=8, device='cpu')
_learn = Learner(_dls, _model, loss_func=nn.CrossEntropyLoss(), lr=1e-2)
_pr = Pruner(_model, 0.5, 'local', large_final, example_inputs=_X[:2], reg=1e-3)
_fit(_learn, GroupRegularizeCallback(_pr))
_pr.prune_model()
test_eq(_model[0].out_channels, 4)

assert _held(_learn) != _live(_model)   # why reset_opt=True: the prune replaced the parameters fastai's optimizer holds
w0 = _model[0].weight.detach().clone()
_fit(_learn, reset_opt=True)
test_eq(_held(_learn), _live(_model))
assert not torch.equal(_model[0].weight, w0)
test_eq(_model(_X[:2]).shape, (2, 4))

In [ ]:
#| slow
# --- resnet18 on the CPU: fit with the callback, prune, fine-tune, save ---
from torchvision.models import resnet18
torch.manual_seed(0)
_rm = resnet18(num_classes=4)
_X, _y = torch.randn(32, 3, 32, 32), torch.randint(0, 4, (32,))
_dls = DataLoaders.from_dsets(TensorDataset(_X[:24], _y[:24]), TensorDataset(_X[24:], _y[24:]), bs=8, device='cpu')
_learn = Learner(_dls, _rm, loss_func=nn.CrossEntropyLoss(), lr=1e-3, path=tempfile.mkdtemp())
_pr = Pruner(_rm, 0.3, 'local', large_final, example_inputs=_X[:2], reg=1e-4)
_probe = _Probe()
_fit(_learn, GroupRegularizeCallback(_pr, schedule=lin), _probe)
assert all(t.isfinite().all() for g in _probe.grads for t in g.values())
n0 = sum(p.numel() for p in _rm.parameters())
_pr.prune_model()
assert sum(p.numel() for p in _rm.parameters()) < n0
_fit(_learn, reset_opt=True)
test_eq(_held(_learn), _live(_rm))
test_eq(_rm(_X[:2]).shape, (2, 4))
assert _learn.save('pruned').exists()